In [2]:
#Import
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor

#Load data
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')

re_train_df = train_df.dropna(axis=0).copy()

#feature engineering
re_train_df['Sex_num'] = re_train_df['Sex'].map({'female': 1, 'male': 0})

re_train_df['Has_cabin'] = re_train_df['Cabin'].notna().astype(int)

re_train_df['FamilySize'] = (
    re_train_df['SibSp'] + re_train_df['Parch'] + 1
)

re_train_df['FarePerPerson'] = (
    re_train_df['Fare'] / re_train_df['FamilySize']
)

re_train_df['IsChild'] = (re_train_df['Age'] < 12).astype(int)

re_train_df['IsElderly'] = (re_train_df['Age'] > 60).astype(int)



#target (what to predict)
y = re_train_df['Survived']
#y = train_df['Survived']

#freatures 
features = [
    "Pclass", "Age", "SibSp", "Parch", "FarePerPerson", "Sex_num", "Has_cabin", "IsChild", "IsElderly"
]

X = re_train_df[features]
#X = train_df[features]

#train split
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 23)

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(train_X, train_y)

#evaluate
xgb_pred = xgb.predict(val_X) 
xgb_mae = mean_absolute_error(val_y, xgb_pred)

#for easier visualisation
print(round(xgb_mae * 100, 5))

28.12445
